In [156]:
import matplotlib.pyplot as plt
import astropy.units as u
import numpy as np

from utils.vextractor import VEXtractor, parse_vex_time
from utils.delay_file_reader import DelayFileReader
from typing import Tuple, List

In [157]:
SAMPLE_RATE = 32e6
N_SCANS = 17
SCAN_NUMBERS = [f"No{'000' if x < 10 else '00'}{x}" for x in range(1, N_SCANS + 1)]

DELAY_PATH_IB = "/home/matyss/Work/RADIOBLOCKS/E011/delays/E011_Ib.del"
DELAY_PATH_IR = "/home/matyss/Work/RADIOBLOCKS/E011/delays/E011_Ir.del"

CLOCK_OFFSET = {
    "Ir": 0.18501562500000013 * 1e-6,
    "Ib": 0.0 * 1e-6
}

CLOCK_RATE = {
    "Ir": -1.66e-7 * 1e-6,
    "Ib": 2.37e-13 * 1e-6
}

CLOCK_REF = parse_vex_time("2024y121d03h51m15s")

vex = VEXtractor("/home/matyss/Work/RADIOBLOCKS/E011/E011.vix")

In [158]:
def read_delays(file: str, scan_name: str) -> Tuple[List[float], List[float]]:
    reader = DelayFileReader(file)
    reader.read_file()

    matching_scans = [scan for scan in reader.scans if scan["scan_name"] == scan_name]

    scan = matching_scans[0]
    sec_of_day = []
    delays = []
    for point in scan["points"]:
        sec_of_day.append(point["sec_of_day"])
        delays.append(point["delay"])

    return sec_of_day, delays

In [159]:
delays = {x: {"Ib": (), "Ir": ()} for x in SCAN_NUMBERS}

for scan in SCAN_NUMBERS:
    delays[scan]["Ib"] = read_delays(DELAY_PATH_IB, scan)
    delays[scan]["Ir"] = read_delays(DELAY_PATH_IR, scan)

In [160]:
print("Relative delay in samples\n")

header = f"{'Scan':<8} {'First':>10} {'Last':>10} {'Points':>10}"
print(header)
print("-" * len(header))

for scan in SCAN_NUMBERS:
    first = (delays[scan]["Ir"][1][1] - delays[scan]["Ib"][1][1]) * SAMPLE_RATE
    last = (delays[scan]["Ir"][1][-1] - delays[scan]["Ib"][1][-1]) * SAMPLE_RATE
    n_points = len(delays[scan]["Ir"][1])

    print(f"{scan:<8} {first:>10.2f} {last:>10.2f} {n_points:>10}")

Relative delay in samples

Scan          First       Last     Points
-----------------------------------------
No0001       -31.24     -31.35        183
No0002        22.61      22.15        183
No0003        -0.77      -0.97        183
No0004         8.93       8.85        183
No0005       -40.83     -39.95        183
No0006       -55.93     -53.75        542
No0007       -53.32     -51.05        542
No0008       -50.73     -48.39        542
No0009       -31.06     -30.12        183
No0010       -47.72     -45.27        542
No0011       -44.81     -42.29        542
No0012       -41.95     -39.36        542
No0013       -20.71     -19.73        183
No0014       -38.58     -35.90        542
No0015       -35.43     -32.69        542
No0016       -32.35     -29.56        542
No0017       -10.00      -8.99        183


In [161]:
clock_ref_sod = (
    CLOCK_REF.datetime.hour * 3600
    + CLOCK_REF.datetime.minute * 60
    + CLOCK_REF.datetime.second
    + CLOCK_REF.datetime.microsecond / 1e6
)

print(f"CLOCK_REF SOD: {clock_ref_sod:.6f} s\n")

header = f"{'SCAN':<8} {'ST':<4} {'MEAN |Δdelay| (s)':>20}"
print(header)
print("-" * len(header))

for scan in SCAN_NUMBERS:
    scan_start = vex.start_time(scan)
    scan_start_sod = (
        scan_start.datetime.hour * 3600
        + scan_start.datetime.minute * 60
        + scan_start.datetime.second
        + scan_start.datetime.microsecond / 1e6
    )

    for station in ["Ib", "Ir"]:
        sod = np.asarray(delays[scan][station][0])
        delta_sec = sod - scan_start_sod

        t_abs = scan_start + delta_sec * u.s
        sec_clock = (t_abs - CLOCK_REF).to_value(u.s)

        delay_orig = np.asarray(delays[scan][station][1])
        delay_corr = (
            delay_orig
            + CLOCK_OFFSET[station]
            + sec_clock * CLOCK_RATE[station]
        )

        diff = np.mean(np.abs(delay_corr - delay_orig))

        print(f"{scan:<8} {station:<4} {diff:>20.6e}")

    print()

CLOCK_REF SOD: 13875.000000 s

SCAN     ST      MEAN |Δdelay| (s)
----------------------------------
No0001   Ib           8.010631e-16
No0001   Ir           1.855767e-07

No0002   Ib           7.515239e-16
No0002   Ir           1.855420e-07

No0003   Ib           7.017193e-16
No0003   Ir           1.855072e-07

No0004   Ib           6.522181e-16
No0004   Ir           1.854725e-07

No0005   Ib           5.813599e-16
No0005   Ir           1.854228e-07

No0006   Ib           4.890512e-16
No0006   Ir           1.853582e-07

No0007   Ib           3.544309e-16
No0007   Ir           1.852639e-07

No0008   Ib           2.195801e-16
No0008   Ir           1.851694e-07

No0009   Ib           1.203974e-16
No0009   Ir           1.851000e-07

No0010   Ib           3.835915e-17
No0010   Ir           1.850355e-07

No0011   Ib           1.065319e-16
No0011   Ir           1.849410e-07

No0012   Ib           2.411458e-16
No0012   Ir           1.848467e-07

No0013   Ib           3.405556e-16
No0013   Ir 